# 24 — Free Evolution & Tomography

Consolidates legacy experiments 78, 79, 80, 81, and 82.

1. **Free evolution of cavity state** — let prepared state evolve and track via Wigner (Exp 78)
2. **Time-resolved Wigner** — Wigner tomography at multiple evolution times (Exp 79)
3. **Cavity T1 with state prep** — decay of prepared-state populations (Exp 80)
4. **Photon number parity evolution** — track parity vs. time (Exp 81)
5. **Joint qubit-cavity tomography** — full joint state reconstruction (Exp 82)

## 1. Shared Session Bootstrap

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from qualang_tools.units import unit

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "qubox").exists():
    REPO_ROOT = Path(r"E:\qubox")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from qubox.compat.notebook import (
    StorageWignerTomography,
    load_stage_checkpoint,
    open_notebook_stage,
    save_stage_checkpoint,
)

REGISTRY_BASE = Path(r"E:\qubox")
SAMPLE_ID = "post_cavity_sample_A"
COOLDOWN_ID = "cd_2025_02_22"
QOP_IP = "10.157.36.68"
CLUSTER_NAME = "Cluster_2"

stage = open_notebook_stage(
    stage_name="24_free_evolution_tomography",
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    qop_ip=QOP_IP,
    cluster_name=CLUSTER_NAME,
    force_reopen=True,
    close_existing=True,
)
session = stage.session
attr = stage.attr
SESSION_BOOTSTRAP_PATH = stage.bootstrap_path
u = unit(coerce_to_integer=True)

state_prep_checkpoint = load_stage_checkpoint(
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    stage_name="23_quantum_state_preparation",
)

print(f"Repo root on sys.path: {REPO_ROOT}")
print(f"Shared session bootstrap: {SESSION_BOOTSTRAP_PATH}")
print(f"Stage checkpoint path: {stage.checkpoint_path}")
print(f"QM endpoint: {QOP_IP} ({CLUSTER_NAME})")
if state_prep_checkpoint is not None:
    print(f"Prior stage 23 status: {state_prep_checkpoint['status']}")

## 2. Free Evolution Defaults

In [ ]:
# Evolution time sweep
T_EVOLVE_US = [0, 1, 5, 10, 20, 50, 100]

# Wigner settings
WIGNER_ALPHA_MAX = 3.0
WIGNER_ALPHA_STEP = 0.1
WIGNER_N_AVG = 10000

# General
FREE_EVOL_N_AVG = 10000

print("Free evolution settings:")
print(f"  t_evolve (μs): {T_EVOLVE_US}")
print(f"  Wigner: α_max={WIGNER_ALPHA_MAX}, dα={WIGNER_ALPHA_STEP}")
print(f"  n_avg: {FREE_EVOL_N_AVG}")

## 3. Free Evolution of Cavity State — Exp 78

Prepare a cavity state, let it evolve freely, and measure qubit response.

In [ ]:
free_evol_result = session.cavity_free_evolution(
    t_evolve_us=T_EVOLVE_US,
    n_avg=FREE_EVOL_N_AVG,
)

print("Free evolution measurement complete.")

## 4. Time-Resolved Wigner Tomography — Exp 79

Wigner function at multiple evolution times.

In [ ]:
wigner_time_results = {}

for t_us in T_EVOLVE_US:
    wigner = StorageWignerTomography(session)
    result = wigner.run(
        alpha_max=WIGNER_ALPHA_MAX,
        alpha_step=WIGNER_ALPHA_STEP,
        n_avg=WIGNER_N_AVG,
        t_evolve_us=t_us,
    )
    analysis = wigner.analyze(result, update_calibration=False)
    wigner.plot(analysis)
    wigner_time_results[t_us] = analysis
    print(f"  t={t_us} μs: done")

print("Time-resolved Wigner tomography complete.")

## 5. Cavity T1 with State Prep — Exp 80

Measure population decay of a prepared cavity state.

In [ ]:
cavity_t1_result = session.cavity_t1_with_state_prep(
    t_evolve_us=T_EVOLVE_US,
    n_avg=FREE_EVOL_N_AVG,
)

print("Cavity T1 with state prep complete.")

## 6. Photon Number Parity Evolution — Exp 81

Track parity of the cavity photon number as a function of time.

In [ ]:
parity_result = session.photon_parity_evolution(
    t_evolve_us=T_EVOLVE_US,
    n_avg=FREE_EVOL_N_AVG,
)

print("Photon number parity evolution complete.")

## 7. Joint Qubit-Cavity Tomography — Exp 82

Full joint state reconstruction of the qubit-cavity system.

In [ ]:
joint_tomo_result = session.joint_qubit_cavity_tomography(
    n_avg=FREE_EVOL_N_AVG,
)

print("Joint qubit-cavity tomography complete.")

## 8. Save Checkpoint

In [ ]:
stage_checkpoint_path = save_stage_checkpoint(
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    stage_name="24_free_evolution_tomography",
    status="characterized",
    summary="Free evolution, time-resolved Wigner, cavity T1, parity evolution, and joint tomography.",
    consumed_inputs={
        "t_evolve_us": T_EVOLVE_US,
        "wigner_alpha_max": WIGNER_ALPHA_MAX,
        "wigner_n_avg": WIGNER_N_AVG,
        "free_evol_n_avg": FREE_EVOL_N_AVG,
    },
    persisted_outputs={},
    advisory_outputs={},
    next_stage="25_context_aware_sqr_calibration",
    notes=[
        "All experiments in this notebook are characterization-only.",
        "Time-resolved Wigner data is useful for extracting decoherence rates.",
    ],
    metrics={},
)

print(f"Stage checkpoint saved to: {stage_checkpoint_path}")